# Baselines and input error (January 2024)

Two checks on one month of data, quick enough to rerun:

1. **How much does XGBoost add over a formula?** Metered fares are mostly distance and time, so almost any model gets a high R². The useful number is the error reduction against simple baselines on the same chronological split.
2. **How sensitive is it to its inputs?** The model learns from each trip's *recorded* distance and duration, but a live request only has an estimate from Google Routes or a straight-line guess. Perturbing the inputs shows how fast the error grows.

The training notebook (`03_training.ipynb`) repeats the baseline comparison on the full-year holdout.

In [1]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score
from sklearn.model_selection import KFold
from sklearn.preprocessing import TargetEncoder

from taxi_data import load_hourly_weather, load_trips, load_zones, one_hot_boroughs

zones = load_zones()
trips = one_hot_boroughs(load_trips(1, zones, load_hourly_weather()))
cut = int(len(trips) * 0.8)
train, test = trips.iloc[:cut].copy(), trips.iloc[cut:].copy()
print(f"{len(trips):,} trips: train {len(train):,}, test {len(test):,} "
      f"({test.tpep_pickup_datetime.min():%b %d} to {test.tpep_pickup_datetime.max():%b %d})")

2,827,048 trips: train 2,261,638, test 565,410 (Jan 26 to Jan 31)


## 1. Baselines

Two versions of the meter formula, because the difference matters for the API's fallback (used for routes with no fare history). The TLC meter charges $0.70 per fifth of a mile while the cab moves faster than 12 mph and $0.70 per minute while it is slower, never both at once. A formula that adds the full time charge on top of the distance charge for slow trips overcharges in traffic.

In [2]:
y = test.fare_amount.to_numpy()
results = []

def score(name, pred):
    results.append({
        "Method": name,
        "MAE ($)": round(mean_absolute_error(y, pred), 2),
        "MAPE (%)": round(100 * mean_absolute_percentage_error(y, pred), 1),
        "R²": round(r2_score(y, pred), 3),
    })

minutes = test.trip_time / 60
slow = test.implied_speed_mph < 12
score("Meter, distance plus time when under 12 mph", 3.0 + 3.5 * test.trip_miles + np.where(slow, 0.7 * minutes, 0))
score("Meter, 3 + 0.70 × max(5 × miles, minutes)", 3.0 + 0.7 * np.maximum(5 * test.trip_miles, minutes))

linear_cols = ["trip_miles", "trip_time", "is_airport_trip"]
linear = LinearRegression().fit(train[linear_cols], train.fare_amount)
score("Linear regression (miles, seconds, airport flag)", linear.predict(test[linear_cols]))
pd.DataFrame(results)

,Method,MAE ($),MAPE (%),R²
0,"Meter, distance plus time when under 12 mph",4.64,29.8,0.793
1,"Meter, 3 + 0.70 × max(5 × miles, minutes)",2.10,10.3,0.921
2,"Linear regression (miles, seconds, airport flag)",1.51,9.6,0.946


## 2. XGBoost with the training notebook's features

Same features and out-of-fold route encoding as `03_training.ipynb`, with fixed hyperparameters (300 trees, depth 6, learning rate 0.05) instead of a search.

In [3]:
encoder = TargetEncoder(target_type="continuous", smooth=50.0, cv=KFold(n_splits=5, shuffle=True, random_state=42))
train["historical_route_fare"] = encoder.fit_transform(train[["route_id"]], train.fare_amount).ravel()
test["historical_route_fare"] = encoder.transform(test[["route_id"]]).ravel()

borough_cols = [c for c in trips.columns if c.startswith(("PU_Borough_", "DO_Borough_"))]
features = (["trip_miles", "trip_time", "request_hour", "is_weekend", "is_airport_trip",
             "temp", "precip", "snow", "implied_speed_mph"] + borough_cols + ["historical_route_fare"])

model = xgb.XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.05, subsample=0.9,
                         colsample_bytree=0.8, tree_method="hist", n_jobs=-1, random_state=42)
model.fit(train[features], train.fare_amount)
score("XGBoost (recorded distance and time)", model.predict(test[features]))
pd.DataFrame(results)

,Method,MAE ($),MAPE (%),R²
0,"Meter, distance plus time when under 12 mph",4.64,29.8,0.793
1,"Meter, 3 + 0.70 × max(5 × miles, minutes)",2.10,10.3,0.921
2,"Linear regression (miles, seconds, airport flag)",1.51,9.6,0.946
3,XGBoost (recorded distance and time),0.75,4.5,0.975


## 3. Input error

Each test trip's duration and distance are scaled by a uniform random error (±15% / ±10%, then ±25% / ±20%), speed is recomputed, and the same model is scored again.

In [4]:
rng = np.random.default_rng(0)
for time_error, distance_error in [(0.15, 0.10), (0.25, 0.20)]:
    noisy = test.copy()
    noisy["trip_time"] = test.trip_time * rng.uniform(1 - time_error, 1 + time_error, len(test))
    noisy["trip_miles"] = test.trip_miles * rng.uniform(1 - distance_error, 1 + distance_error, len(test))
    noisy["implied_speed_mph"] = (noisy.trip_miles / noisy.trip_time * 3600).clip(0, 75)
    score(f"XGBoost, ±{time_error:.0%} time and ±{distance_error:.0%} distance error", model.predict(noisy[features]))

summary = pd.DataFrame(results)
summary

,Method,MAE ($),MAPE (%),R²
0,"Meter, distance plus time when under 12 mph",4.64,29.8,0.793
1,"Meter, 3 + 0.70 × max(5 × miles, minutes)",2.10,10.3,0.921
2,"Linear regression (miles, seconds, airport flag)",1.51,9.6,0.946
3,XGBoost (recorded distance and time),0.75,4.5,0.975
4,"XGBoost, ±15% time and ±10% distance error",1.04,6.3,0.971
5,"XGBoost, ±25% time and ±20% distance error",1.44,8.5,0.962


In [5]:
linear_mae = summary.loc[summary.Method.str.startswith("Linear"), "MAE ($)"].item()
xgb_mae = summary.loc[summary.Method == "XGBoost (recorded distance and time)", "MAE ($)"].item()
print(f"XGBoost cuts MAE by {1 - xgb_mae / linear_mae:.0%} against the linear baseline (${linear_mae:.2f} -> ${xgb_mae:.2f}).")

XGBoost cuts MAE by 50% against the linear baseline ($1.51 -> $0.75).
